# Puzzle

https://thefiddler.substack.com/p/can-you-gerrymander-like-a-dummy

### This Week’s Fiddler
Fiddler Nation has an upcoming election between its two major political parties: the Suns and the Moons. You are drawing a new electoral map, and you have been tasked with gerrymandering on behalf of your party, the Moons. Fortunately, you know exactly how every citizen will vote in the election.

Fiddler Nation has seven districts, each with an equal population. For simplicity, assume that the Moons win any district in which they receive at least half of the vote.

What is the smallest total fraction of Fiddler Nation’s population needed to vote Moon such that it’s possible for the Moons to win a majority of the seven districts?

### This Week’s Extra Credit
In the next election cycle you’re back at it, gerrymandering on behalf of the Moon party once more. This time, you can draw a map that results in any set of probabilities P1, P2, P3, P4, P5, P6, and P7 that you’ll win each of the corresponding districts—subject to the constraint that the average of the probabilities is 1/2. In other words, P1 + P2 + P3 + P4 + P5 + P6 + P7 = 7/2.

The bosses of the Moon party want to go big. More specifically, they want to maximize the party’s chances of winning at least five of the seven districts. However, you are concerned that such a greedy attempt at gerrymandering may backfire, a phenomenon that has become known as dummymandering. When you draw a map that maximizes your chances of winning at least five districts, the probability the Moon party blows it and winds up losing a majority of the seven districts is LA.

Your personal preference is to simply maximize the party’s chances of winning at least four of the seven districts—a simple majority of them. When you draw a map that maximizes these chances, the probability the Moon party loses a majority of the seven districts is LB.

How much greater than LB is LA?

# Fiddler Solution

I think this happens when the Moons win 4 districts by getting 0.5 + eps of the vote in those districts and they lose the other districts completely (win 0% of the vote there).

So, the Moons vote fraction is 4(0.5 + eps) / 7 = 2/7 + epsilon.

So, the Fiddler answer is 2/7 + epsilon = 0.285714 + epsilon.

# Extra Credit Solution

It's possible that the optimal strategy is simple. Put all your probability in the 4 or 5 districts you want to win and 0 elsewhere. It's easy to calculate the values for those cases, but I don't have any reasoning to show that this is indeed optimal. So, we need to search the full space of possibilities.

Some [googling](https://www.google.com/search?q=how+to+write+a+function+for+numpy+minimization+with+constraints&sca_esv=38d47f4918aae668&sxsrf=APpeQnu0gr9Mb7wODCcpNJsOdeHjPOd7gg%3A1792262511785&source=hp&ei=b8HTauCELYzpmLQP8fO_0AU&iflsig=ABILxe8AAAAAatPPf3p0tk15ot_LNrZ6GBCrDfepDiBr&ved=0ahUKEwig-OyS2sGXAxWMNIYAHfH5D1oQ4dUDCD4&uact=5&oq=how+to+write+a+function+for+numpy+minimization+with+constraints&gs_lp=Egdnd3Mtd2l6Ij9ob3cgdG8gd3JpdGUgYSBmdW5jdGlvbiBmb3IgbnVtcHkgbWluaW1pemF0aW9uIHdpdGggY29uc3RyYWludHNI1i9QAFjmLHABeACQAQCYAaEBoAHNEKoBBDE2Lja4AQPIAQD4AQGYAhSgAqQPwgINECMYogcYngYY8AUYJ8ICBBAjGCfCAg0QABiABBiKBRhDGLQHwgIKEAAYgAQYigUYQ8ICCxAAGIAEGLEDGIMBwgIOEC4YgAQYsQMYxwEY0QPCAg0QIxjwBRieBhiiBxgnwgIOEAAYgAQYigUYkQIYtAfCAgsQABiABBiKBRiRAsICBRAhGKABwgIFECEYnwXCAgUQIRirAsICBBAhGAqYAwCSBwUxMC4xMKAHzbIBsgcEOS4xMLgHog_CBwY4LjExLjHIBxmACAE&sclient=gws-wiz) has produced instructions on how to do minimization with constraints.

In [66]:
import numpy as np
from scipy.optimize import minimize
from itertools import product

def probs_of_wins(district_win_probabilities):
    N = len(district_win_probabilities)
    
    p_i = np.zeros(N + 1)  # Initialize probabilities for i = 0 to N

    if (any(p < 0 or p > 1 for p in district_win_probabilities)):
        return p_i  # Return zero probabilities if any input probability is invalid

    for outcomes in product([0, 1], repeat=N):  # Generate all combinations of wins/losses
        i = sum(outcomes)  # Count the number of wins (1s)
        prob = 1
        for j, outcome in enumerate(outcomes):
            prob *= (district_win_probabilities[j] if outcome == 1 else (1 - district_win_probabilities[j]))
        p_i[i] += prob  # Accumulate the probability for i wins

    assert np.isclose(sum(p_i), 1), f"Probabilities do not sum to 1. Sum: {sum(p_i)}. Probabilities: {p_i}"
    return p_i  # Return the list of probabilities for i = 0 to N

u_results= {}
for i in range(4,8):
    u_probabilities = [7.0/(2*i)]*i + [0]*(7-i)
    u_results[i] = probs_of_wins(u_probabilities)
    print(f"Probabilities of k wins u{i} case: {u_results[i]}")

Probabilities of k wins u4 case: [2.44140625e-04 6.83593750e-03 7.17773438e-02 3.34960938e-01
 5.86181641e-01 0.00000000e+00 0.00000000e+00 0.00000000e+00]
Probabilities of k wins u5 case: [0.00243 0.02835 0.1323  0.3087  0.36015 0.16807 0.      0.     ]
Probabilities of k wins u6 case: [0.00523278 0.04395536 0.15384376 0.28717502 0.30153377 0.16885891
 0.03940041 0.        ]
Probabilities of k wins u7 case: [0.0078125 0.0546875 0.1640625 0.2734375 0.2734375 0.1640625 0.0546875
 0.0078125]


In [67]:
def prob_at_least_k_wins(district_win_probabilities, k=4):
    p_wins = probs_of_wins(district_win_probabilities)
    at_least_k_wins = np.sum(p_wins[k:])  # Sum probabilities for i=k to N
    return at_least_k_wins

In [68]:
def objective_4(district_win_probabilities):
    # Calculate the probability of at least 4 wins
    prob = prob_at_least_k_wins(district_win_probabilities, k=4)

    # We want to maximize this probability, so we minimize its negative
    return -prob

def objective_5(district_win_probabilities):
    # Calculate the probability of at least 5 wins
    prob = prob_at_least_k_wins(district_win_probabilities, k=5)

    # We want to maximize this probability, so we minimize its negative
    return -prob

In [69]:
# inequality constraint function to ensure probabilities are valid
def valid_probabilities_constraint(district_win_probabilities):
    district_loss_probabilities = [1 - p for p in district_win_probabilities]
    lower_probs = np.minimum(district_win_probabilities, district_loss_probabilities)
    return np.min(lower_probs)  # Should be >= 0 if all probabilities are valid
    
# equality constraint to ensure the probabilities average to 0.5
def average_probability_constraint(district_win_probabilities):
    average_probability = np.mean(district_win_probabilities)
    return average_probability - 0.5  # Should be == 0 if average is 0.5

# inequality constraint to ensure probabilities are in non-increasing order
def ordering_constraint(district_win_probabilities):
    diffs = -np.diff(district_win_probabilities)
    return np.min(diffs)  # Should be >= 0 if probabilities are in non-increasing order

In [70]:
# Define the constraints
constraints = [
    {'type': 'ineq', 'fun': valid_probabilities_constraint},
    {'type': 'eq', 'fun': average_probability_constraint},
    {'type': 'ineq', 'fun': ordering_constraint},
]


In [71]:
def minimize_from_many_start_points(obj_fn, random_starts=10000):
    best_fun = np.inf
    best_x = None
    best_src = None
    for i in range(4,8+random_starts):
        # Initial guess for the probabilities
        if i <= 7:
            initial_guess = np.zeros(7)
            initial_guess[:i] = 1  # Set the first i districts to 1
        else:
            initial_guess = np.random.uniform(0, 1, 7)  # Random initial guess for all 7 districts
        # Normalize initial guess to have an average of 0.5
        initial_guess = initial_guess / np.mean(initial_guess) * 0.5
        initial_guess.sort(descending=True)  # Sort to help with the ordering constraint

        ig_fun = obj_fn(initial_guess)
        if ig_fun < best_fun:
            best_fun = ig_fun
            best_x = initial_guess
            best_src = f"Initial Guess {i}"

        # Perform the optimization
        result = minimize(
            obj_fn,
            initial_guess,
            constraints=constraints
        )
        if result.success:
            if result.fun < best_fun:
                best_fun = result.fun
                best_x = result.x
                best_src = f"Minimization {i}"

    print (f"Min value found: {best_fun} at {best_x} from {best_src}")
    return best_fun, best_x, best_src

In [72]:
results_4 = minimize_from_many_start_points(objective_4, random_starts=10000)

Min value found: -0.586181640625 at [0.875 0.875 0.875 0.875 0.    0.    0.   ] from Initial Guess 4


In [73]:
results_5 = minimize_from_many_start_points(objective_5, random_starts=10000)

Min value found: -0.2265625 at [0.5 0.5 0.5 0.5 0.5 0.5 0.5] from Initial Guess 7


So, after all this search, the optimal choices are from the original few guesses. Let's tabulate that.

In [74]:
for i in range(4, 8):
    prob_k_wins = u_results[i]
    prob_at_most_3  = np.sum(prob_k_wins[:4])  # Sum probabilities for i=0 to 3
    prob_at_least_4 = np.sum(prob_k_wins[4:]) # Sum probabilities for i=4 to N
    prob_at_least_5 = np.sum(prob_k_wins[5:]) # Sum probabilities for i=5 to N
    print(f"uniform over {i} districts, 0 elsewhere | P(3 or fewer wins): {prob_at_most_3:.8f} | P(4 or more wins): {prob_at_least_4:.8f} | P(5 or more wins): {prob_at_least_5:.8f}")

uniform over 4 districts, 0 elsewhere | P(3 or fewer wins): 0.41381836 | P(4 or more wins): 0.58618164 | P(5 or more wins): 0.00000000
uniform over 5 districts, 0 elsewhere | P(3 or fewer wins): 0.47178000 | P(4 or more wins): 0.52822000 | P(5 or more wins): 0.16807000
uniform over 6 districts, 0 elsewhere | P(3 or fewer wins): 0.49020691 | P(4 or more wins): 0.50979309 | P(5 or more wins): 0.20825932
uniform over 7 districts, 0 elsewhere | P(3 or fewer wins): 0.50000000 | P(4 or more wins): 0.50000000 | P(5 or more wins): 0.22656250


So, we can see that:

* LA, the losing probability when you try to maximize the probability of winning 5, is 0.5
* LB, the losing probability when you try to maximize the probability of winning 4, is 0.41381836

So, the difference (LA - LB) = 0.5 - .41381836 = 0.08618164

# Conclusions

Fiddler Answer = 2/7 + epsilon = 0.285714 + epsilon.

Extra Credit Answer = LA - LB = 0.08618164